# DAX Query View Tests — DEV (Power BI Desktop)

Runs PQL.Assert DAX Query View suites against the currently-open Power BI Desktop instance for
this project. Uses the shared `dax_test_helpers.py`/`run_dax_tests.py` logic — the same
discovery, smoke gate, and result parsing as the CLI and pytest wrapper.

Fill in `MODEL_DAX_QUERIES_DIR` below before running.

In [ ]:
import sys
from pathlib import Path

# --- Explicit configuration (no hard-coded model name) ---
SCRIPTS_DIR = Path("../scripts").resolve()  # adjust if this notebook is copied elsewhere
MODEL_DAX_QUERIES_DIR = Path("REPLACE_WITH_YOUR_MODEL/DAXQueries")
PROFILE = "DEV"
FILENAME_FILTER = None  # e.g. "MeasureCertification"
ENVIRONMENT_FILTER = None  # "ANY", "DEV", or "PROD"
REPORTS_DIR = Path("reports")

sys.path.insert(0, str(SCRIPTS_DIR))
import dax_test_helpers as h
import run_dax_tests as r

In [ ]:
transport = r.build_transport(PROFILE)
smoke_failure = r.run_smoke_gate(transport)
if smoke_failure is not None:
    raise RuntimeError(f"Smoke gate failed [{smoke_failure.error_type}]: {smoke_failure.error_message}")
print("Smoke gate passed.")

In [ ]:
test_files = h.discover_test_files(MODEL_DAX_QUERIES_DIR, name_filter=FILENAME_FILTER, environment=ENVIRONMENT_FILTER)
print(f"Discovered {len(test_files)} test file(s): {[f.name for f in test_files]}")
results = r.run_suite(transport, test_files)
failed = [x for x in results if x.passed is not True]
print(f"Ran {len(results)} assertion(s); {len(failed)} failure(s).")
for x in failed:
    print(f"  [{x.error_type}] {x.test_file} :: {x.test_name} - {x.error_message}")

In [ ]:
r.write_junit(REPORTS_DIR / "junit.xml", results)
r.write_markdown(REPORTS_DIR / "failures.md", results)
print(f"Wrote reports to {REPORTS_DIR.resolve()}")